# BoW sweep analysis

Stateful notebook: one cell per sweep loads a variable; each plot is its own cell so you can tweak `y_exprs`, `show_seed_bar`, or `x_scale` independently. `plot_vs_epoch` draws per-epoch traces (for rollout sweeps, colored by rollouts, legend-grouped by corr, with only the highest-corr group visible by default — click any other group in the legend to reveal it). `plot_vs_corr` picks the argmax-over-epochs point per study and plots it against the dataset correlation.

In [1]:
from src import get_repo_base
from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
)

ARTIFACTS = get_repo_base() / "artifacts"

Y_EXPRS = [
    corr_expr(split="train", y="ground_truth"),
    corr_expr(split="val", y="ground_truth"),
]


def describe(label, analysis):
    if analysis is None:
        print(f"{label:<32s}  no artifacts")
        return
    n_runs = sum(len(v) for v in analysis.studies.values())
    n_groups = len(analysis.studies)
    max_seeds = max(len(v) for v in analysis.studies.values())
    print(
        f"{label:<32s}  {n_runs:>4d} runs   {n_groups:>3d} groups   "
        f"up to {max_seeds} seeds/group"
    )

### Supervised-learning

In [2]:
sl = BagOfWordsAnalysisConfig.from_sl_sweep(
    study_base=ARTIFACTS / "bow-sl-sweep",
)
describe("SL", sl)

display(sl.plot_vs_epoch(Y_EXPRS, title="SL: per-epoch", show_seed_bar=True))
display(
    sl.plot_vs_corr(
        Y_EXPRS,
        title="SL: best-epoch vs dataset_corr",
        x_scale="uniform",
        show_seed_bar=True,
    )
)

SL                                  32 runs    16 groups   up to 2 seeds/group


### GRPO artifacts

In [3]:
grpo = BagOfWordsAnalysisConfig.from_grpo_sweep(
    study_base=ARTIFACTS / "bow-grpo-sweep",
)
describe("GRPO", grpo)

display(grpo.plot_vs_epoch(Y_EXPRS, title="GRPO: per-epoch", show_seed_bar=True))
display(
    grpo.plot_vs_corr(
        Y_EXPRS,
        title="GRPO: best-epoch vs dataset_corr",
        x_scale="uniform",
        show_seed_bar=True,
    )
)

GRPO                                44 runs    44 groups   up to 1 seeds/group


### MaxRL artifacts

In [4]:
maxrl_sub = BagOfWordsAnalysisConfig.from_maxrl_sweep(
    study_base=ARTIFACTS / "bow-maxrl-sweep",
    subtract_baseline=True,
)
describe("MaxRL (subtract-baseline)", maxrl_sub)

maxrl_nosub = BagOfWordsAnalysisConfig.from_maxrl_sweep(
    study_base=ARTIFACTS / "bow-maxrl-sweep",
    subtract_baseline=False,
)
describe("MaxRL (no-subtract-baseline)", maxrl_nosub)

MaxRL (subtract-baseline)         no artifacts
MaxRL (no-subtract-baseline)      no artifacts


In [5]:
display(
    maxrl_sub.plot_vs_epoch(
        Y_EXPRS,
        title="MaxRL (subtract-baseline): per-epoch",
        show_seed_bar=True,
    )
)
display(
    maxrl_sub.plot_vs_corr(
        Y_EXPRS,
        title="MaxRL (subtract-baseline): best-epoch vs dataset_corr",
        x_scale="uniform",
        show_seed_bar=True,
    )
)

AttributeError: 'NoneType' object has no attribute 'plot_vs_epoch'

In [ ]:
display(
    maxrl_nosub.plot_vs_epoch(
        Y_EXPRS,
        title="MaxRL (no-subtract-baseline): per-epoch",
        show_seed_bar=True,
    )
)
display(
    maxrl_nosub.plot_vs_corr(
        Y_EXPRS,
        title="MaxRL (no-subtract-baseline): best-epoch vs dataset_corr",
        x_scale="uniform",
        show_seed_bar=True,
    )
)